In [1]:
import os
import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import create_engine, text
import requests
import sqlite3

In [2]:
url = (
  "https://raw.githubusercontent.com/"
  "routineactivity/adhoc_notebooks/main/"
  "crime_text_summaries/nyc_crime.sqlite"
)
r = requests.get(url)
r.raise_for_status()

with open("nyc_crime.sqlite", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("nyc_crime.sqlite")
df  = pd.read_sql("SELECT * FROM nypd_recent_felonies_data", conn)
conn.close()

In [3]:
df.head()

,geom,cmplnt_num,cmplnt_fr_dt,cmplnt_fr_tm_time,cmplnt_to_dt,cmplnt_to_tm,addr_pct_cd,rpt_dt,ky_cd,ofns_desc,...,susp_sex,susp_race,parks_nm,hadevelopt,housing_psa,station_name,x_coord_cd,y_coord_cd,latitude,longitude
0,0101000020D70800000000000024D42E410000000030E2...,298131818,2024-12-17,15:15:00.000000,2024-12-17,15:30:00.000000,44,2024-12-17,105,ROBBERY,...,None,UNKNOWN,CLAREMONT PARK,None,None,None,1010194,244806,40.838583,-73.906239
1,0101000020D708000000000000C2F82D41000000008086...,297524958,2024-12-05,06:25:00.000000,2024-12-05,07:00:00.000000,6,2024-12-05,107,BURGLARY,...,None,UNKNOWN,None,None,None,None,982113,209104,40.740619,-74.007711
2,0101000020D708000000000000C8FD2D4100000000D8A6...,297994260,2024-12-14,18:52:00.000000,2024-12-14,18:56:00.000000,10,2024-12-14,105,ROBBERY,...,None,BLACK,None,None,566,None,982756,210139,40.743470,-74.005392
3,0101000020D708000000000000E0E92D4100000000F068...,298515531,2024-12-26,23:30:00.000000,2024-12-27,05:30:00.000000,68,2024-12-27,107,BURGLARY,...,None,(null),None,None,None,None,980208,167198,40.625604,-74.014560
4,0101000020D708000000000000BC4A2E4100000000902F...,298393680,2024-12-22,21:32:00.000000,2024-12-22,21:58:00.000000,60,2024-12-23,107,BURGLARY,...,None,(null),None,None,None,None,992606,148978,40.575582,-73.969921


In [4]:
df.columns

Index(['geom', 'cmplnt_num', 'cmplnt_fr_dt', 'cmplnt_fr_tm_time',
       'cmplnt_to_dt', 'cmplnt_to_tm', 'addr_pct_cd', 'rpt_dt', 'ky_cd',
       'ofns_desc', 'pd_cd', 'pd_desc', 'crm_atpt_cptd_cd', 'law_cat_cd',
       'boro_nm', 'patrol_boro', 'loc_of_occur_desc', 'prem_type_desc',
       'juris_desc', 'jurisdiction_code', 'vic_age_group', 'vic_sex',
       'vic_race', 'susp_age_group', 'susp_sex', 'susp_race', 'parks_nm',
       'hadevelopt', 'housing_psa', 'station_name', 'x_coord_cd', 'y_coord_cd',
       'latitude', 'longitude'],
      dtype='object')

In [5]:
# parse dates and times
df['cmplnt_fr_dt'] = pd.to_datetime(df['cmplnt_fr_dt'], errors='coerce')
df['cmplnt_fr_tm_time'] = pd.to_datetime(df['cmplnt_fr_tm_time'],format='%H:%M:%S.%f',errors='coerce')
df['cmplnt_fr_tm_time'] = df['cmplnt_fr_tm_time'].dt.time
# extract hour
df['hour'] = pd.to_datetime(df['cmplnt_fr_tm_time'].astype(str), format='%H:%M:%S', errors='coerce').dt.hour

# derive day of week
df['day_of_week'] = df['cmplnt_fr_dt'].dt.day_name()

# derive 3-hour periods
bins = list(range(0, 25, 3)) 
labels = [f"{b:02d}:00-{(b+2):02d}:59" for b in bins[:-1]]
df['period'] = pd.cut(df['hour'], bins=bins, right=False, labels=labels)

# helper to get top-3 from any summary
def top_n(summary_df, label_col):
    return [(getattr(r, label_col), r.count, r.percent) for r in summary_df.head(3).itertuples()]

# loop through boroughs and crime categories
boroughs = df['boro_nm'].dropna().unique()
crimes = df['ofns_desc'].dropna().unique()

for borough in boroughs:
    for crime in crimes:
        subset = df[(df['boro_nm'] == borough) & (df['ofns_desc'] == crime)]
        if subset.empty:
            continue

        # counts for days, periods, etc
        day_counts    = subset['day_of_week'].value_counts()
        period_counts = subset['period'].value_counts().sort_values(ascending=False)
        attempt_counts = subset['crm_atpt_cptd_cd'].value_counts()
        prem_counts    = subset['prem_type_desc'].value_counts()
        pct_counts     = subset['addr_pct_cd'].value_counts()

        # attempt classification
        total_attempts    = len(subset)
        completed_count   = (subset['crm_atpt_cptd_cd'] == 'COMPLETED').sum()
        completed_percent = completed_count / total_attempts

        # victim distributions
        vic_age_counts   = subset['vic_age_group'].value_counts()
        vic_gender_counts = subset['vic_sex'].value_counts()
        vic_race_counts  = subset['vic_race'].value_counts()

        # suspect known distributions
        enum_ages = ['<18','18-24','25-44','45-64','65+']
        sus_age_known_counts   = subset.loc[subset['susp_age_group'].isin(enum_ages), 'susp_age_group'].value_counts()
        sus_gender_known_counts = subset.loc[subset['susp_sex'].isin(['M','F']), 'susp_sex'].value_counts()
        valid_races = ['BLACK','WHITE','WHITE HISPANIC','BLACK HISPANIC',
                       'ASIAN / PACIFIC ISLANDER','AMERICAN INDIAN/ALASKAN NATIVE']
        sus_race_known_counts  = subset.loc[subset['susp_race'].isin(valid_races), 'susp_race'].value_counts()

        # summary dfs
        def make_summary(idx, counts, total):
            return pd.DataFrame({
                idx: counts.index,
                'count': counts.values,
                'percent': counts.values / total
            }).sort_values('count', ascending=False).reset_index(drop=True)

        total_n = len(subset)
        day_summary    = make_summary('day_of_week', day_counts, total_n)
        period_summary = make_summary('period',      period_counts, total_n)
        loc_summary    = make_summary('prem_type',    prem_counts, total_n)
        pct_summary    = make_summary('addr_pct',     pct_counts, total_n)

        # extract top-3
        top_days    = top_n(day_summary,    'day_of_week')
        top_periods = top_n(period_summary, 'period')
        top_locs    = top_n(loc_summary,    'prem_type')
        top_pcts    = top_n(pct_summary,    'addr_pct')

        # victim & suspect percentages
        victim_age_top3 = vic_age_counts.sort_values(ascending=False).head(3).index.tolist()
        
        if len(victim_age_top3) < 3:
            victim_age_top3 += ['Unknown'] * (3 - len(victim_age_top3))


        vic_black_count = (
            vic_race_counts.get('BLACK', 0) + vic_race_counts.get('BLACK HISPANIC', 0))

        
        victim_pct_male  = vic_gender_counts.get('M', 0) / total_n
        victim_pct_black = vic_black_count / total_n
        victim_pct_white = vic_race_counts.get('WHITE', 0) / total_n
        victim_pct_hispanic = vic_race_counts.get('WHITE HISPANIC', 0) / total_n

        suspect_age_top3  = sus_age_known_counts.sort_values(ascending=False).head(3).index.tolist()

        if len(suspect_age_top3) < 3:
            suspect_age_top3 += ['Unknown'] * (3 - len(suspect_age_top3))

        sus_black_count = (
            sus_race_known_counts.get('BLACK', 0) + sus_race_known_counts.get('BLACK HISPANIC', 0))            
    
        suspect_pct_male  = sus_gender_known_counts.get('M', 0) / sus_gender_known_counts.sum()
        suspect_pct_black = sus_black_count / total_n
        suspect_pct_white = sus_race_known_counts.get('WHITE', 0) / sus_race_known_counts.sum()
        suspect_pct_hispanic = sus_race_known_counts.get('WHITE HISPANIC', 0) / sus_race_known_counts.sum()

        # descriptive narrative template
        template = (
            "## Overview of {crime} in {borough}\n\n"
            "There were {total_n} offences reported.\n\n"
            "**Peak days**: {day1} ({day1_n}, {day1_pct:.1%}), {day2} ({day2_n}, {day2_pct:.1%}), {day3} ({day3_n}, {day3_pct:.1%}).\n\n"
            "**Busiest periods**: {time1} ({time1_n}, {time1_pct:.1%}), {time2} ({time2_n}, {time2_pct:.1%}), {time3} ({time3_n}, {time3_pct:.1%}).\n\n"
            "**Offences completed**: ({completed_count}, {completed_percent:.1%}).\n\n"
            "**Top premises**: {loc1} ({loc1_n}, {loc1_pct:.1%}), {loc2} ({loc2_n}, {loc2_pct:.1%}), {loc3} ({loc3_n}, {loc3_pct:.1%}).\n\n"
            "**Top precincts**: {pct1} ({pct1_n}, {pct1_pct:.1%}), {pct2} ({pct2_n}, {pct2_pct:.1%}), {pct3} ({pct3_n}, {pct3_pct:.1%}).\n\n"
            "**Victim** ages: {v_age1}, {v_age2}, {v_age3}; {victim_pct_male:.1%} male; race {victim_pct_black:.1%} Black, {victim_pct_white:.1%} White, {victim_pct_hispanic:.1%} Hispanic.\n\n"
            "**Suspect** ages: {s_age1}, {s_age2}, {s_age3}; {suspect_pct_male:.1%} male; race {suspect_pct_black:.1%} Black, {suspect_pct_white:.1%} White, {suspect_pct_hispanic:.1%} Hispanic."
        )

        # prepare values
        values = {
            'crime': crime,
            'borough': borough,
            'total_n': total_n,
            # days
            'day1': top_days[0][0], 'day1_n': top_days[0][1], 'day1_pct': top_days[0][2],
            'day2': top_days[1][0], 'day2_n': top_days[1][1], 'day2_pct': top_days[1][2],
            'day3': top_days[2][0], 'day3_n': top_days[2][1], 'day3_pct': top_days[2][2],
            # periods
            'time1': top_periods[0][0], 'time1_n': top_periods[0][1], 'time1_pct': top_periods[0][2],
            'time2': top_periods[1][0], 'time2_n': top_periods[1][1], 'time2_pct': top_periods[1][2],
            'time3': top_periods[2][0], 'time3_n': top_periods[2][1], 'time3_pct': top_periods[2][2],
            # completed
            'completed_count':   completed_count,
            'completed_percent': completed_percent,
            # premises
            'loc1': top_locs[0][0], 'loc1_n': top_locs[0][1], 'loc1_pct': top_locs[0][2],
            'loc2': top_locs[1][0], 'loc2_n': top_locs[1][1], 'loc2_pct': top_locs[1][2],
            'loc3': top_locs[2][0], 'loc3_n': top_locs[2][1], 'loc3_pct': top_locs[2][2],
            # precincts
            'pct1': top_pcts[0][0], 'pct1_n': top_pcts[0][1], 'pct1_pct': top_pcts[0][2],
            'pct2': top_pcts[1][0], 'pct2_n': top_pcts[1][1], 'pct2_pct': top_pcts[1][2],
            'pct3': top_pcts[2][0], 'pct3_n': top_pcts[2][1], 'pct3_pct': top_pcts[2][2],
             # victims
            'v_age1': victim_age_top3[0], 'v_age2': victim_age_top3[1], 'v_age3': victim_age_top3[2],
            'victim_pct_male': victim_pct_male, 'victim_pct_black': victim_pct_black,
            'victim_pct_white': victim_pct_white, 'victim_pct_hispanic': victim_pct_hispanic,
            # suspects
            's_age1': suspect_age_top3[0], 's_age2': suspect_age_top3[1], 's_age3': suspect_age_top3[2],
            'suspect_pct_male': suspect_pct_male, 'suspect_pct_black': suspect_pct_black,
            'suspect_pct_white': suspect_pct_white, 'suspect_pct_hispanic': suspect_pct_hispanic,
        }

        # render and display report text
        report = template.format(**values)
        display(Markdown(report))


## Overview of ROBBERY in BRONX

There were 335 offences reported.

**Peak days**: Tuesday (70, 20.9%), Friday (49, 14.6%), Thursday (49, 14.6%).

**Busiest periods**: 15:00-17:59 (63, 18.8%), 18:00-20:59 (61, 18.2%), 12:00-14:59 (56, 16.7%).

**Offences completed**: (303, 90.4%).

**Top premises**: STREET (150, 44.8%), RESIDENCE - APT. HOUSE (70, 20.9%), RESIDENCE - PUBLIC HOUSING (25, 7.5%).

**Top precincts**: 47 (40, 11.9%), 40 (39, 11.6%), 42 (37, 11.0%).

**Victim** ages: 25-44, 45-64, <18; 61.2% male; race 45.1% Black, 3.3% White, 30.1% Hispanic.

**Suspect** ages: 25-44, 18-24, <18; 90.7% male; race 54.6% Black, 3.3% White, 21.0% Hispanic.

## Overview of BURGLARY in BRONX

There were 188 offences reported.

**Peak days**: Tuesday (39, 20.7%), Thursday (31, 16.5%), Saturday (29, 15.4%).

**Busiest periods**: 03:00-05:59 (33, 17.6%), 15:00-17:59 (31, 16.5%), 09:00-11:59 (27, 14.4%).

**Offences completed**: (176, 93.6%).

**Top premises**: RESIDENCE - APT. HOUSE (87, 46.3%), STREET (22, 11.7%), RESIDENCE - PUBLIC HOUSING (13, 6.9%).

**Top precincts**: 44 (27, 14.4%), 40 (23, 12.2%), 42 (20, 10.6%).

**Victim** ages: 25-44, 45-64, 18-24; 30.9% male; race 28.2% Black, 2.7% White, 13.3% Hispanic.

**Suspect** ages: 25-44, 45-64, 18-24; 84.9% male; race 20.2% Black, 6.8% White, 40.5% Hispanic.

## Overview of FELONY ASSAULT in BRONX

There were 600 offences reported.

**Peak days**: Saturday (92, 15.3%), Wednesday (91, 15.2%), Sunday (89, 14.8%).

**Busiest periods**: 21:00-23:59 (94, 15.7%), 15:00-17:59 (93, 15.5%), 18:00-20:59 (93, 15.5%).

**Offences completed**: (570, 95.0%).

**Top premises**: RESIDENCE - APT. HOUSE (277, 46.2%), STREET (130, 21.7%), RESIDENCE - PUBLIC HOUSING (40, 6.7%).

**Top precincts**: 47 (79, 13.2%), 44 (74, 12.3%), 40 (66, 11.0%).

**Victim** ages: 25-44, 45-64, 18-24; 48.7% male; race 56.8% Black, 6.2% White, 25.7% Hispanic.

**Suspect** ages: 25-44, 45-64, 18-24; 73.8% male; race 55.5% Black, 3.1% White, 27.1% Hispanic.

## Overview of ROBBERY in MANHATTAN

There were 238 offences reported.

**Peak days**: Tuesday (45, 18.9%), Sunday (38, 16.0%), Friday (36, 15.1%).

**Busiest periods**: 15:00-17:59 (50, 21.0%), 18:00-20:59 (44, 18.5%), 21:00-23:59 (34, 14.3%).

**Offences completed**: (224, 94.1%).

**Top premises**: STREET (84, 35.3%), RESIDENCE - APT. HOUSE (21, 8.8%), RESIDENCE - PUBLIC HOUSING (18, 7.6%).

**Top precincts**: 23 (25, 10.5%), 14 (19, 8.0%), 18 (19, 8.0%).

**Victim** ages: 25-44, 18-24, 45-64; 57.6% male; race 34.0% Black, 13.0% White, 21.4% Hispanic.

**Suspect** ages: 25-44, 18-24, 45-64; 89.8% male; race 66.4% Black, 5.7% White, 16.7% Hispanic.

## Overview of BURGLARY in MANHATTAN

There were 239 offences reported.

**Peak days**: Thursday (43, 18.0%), Saturday (41, 17.2%), Friday (34, 14.2%).

**Busiest periods**: 03:00-05:59 (45, 18.8%), 00:00-02:59 (39, 16.3%), 09:00-11:59 (28, 11.7%).

**Offences completed**: (232, 97.1%).

**Top premises**: RESIDENCE - APT. HOUSE (78, 32.6%), RESTAURANT/DINER (27, 11.3%), COMMERCIAL BUILDING (26, 10.9%).

**Top precincts**: 19 (29, 12.1%), 6 (23, 9.6%), 14 (20, 8.4%).

**Victim** ages: 25-44, 45-64, 18-24; 20.5% male; race 10.0% Black, 20.1% White, 5.0% Hispanic.

**Suspect** ages: 25-44, 45-64, 18-24; 90.7% male; race 29.7% Black, 20.4% White, 26.1% Hispanic.

...

## Overview of BURGLARY in QUEENS

There were 293 offences reported.

**Peak days**: Friday (52, 17.7%), Wednesday (48, 16.4%), Monday (47, 16.0%).

**Busiest periods**: 03:00-05:59 (57, 19.5%), 15:00-17:59 (43, 14.7%), 18:00-20:59 (42, 14.3%).

**Offences completed**: (271, 92.5%).

**Top premises**: RESIDENCE-HOUSE (107, 36.5%), RESIDENCE - APT. HOUSE (53, 18.1%), STREET (23, 7.8%).

**Top precincts**: 109 (38, 13.0%), 111 (37, 12.6%), 107 (28, 9.6%).

**Victim** ages: 45-64, 25-44, 65+; 36.5% male; race 7.2% Black, 13.3% White, 10.9% Hispanic.

**Suspect** ages: 25-44, 45-64, 18-24; 94.4% male; race 12.6% Black, 13.6% White, 29.6% Hispanic.

## Overview of FELONY ASSAULT in QUEENS

There were 427 offences reported.

**Peak days**: Sunday (83, 19.4%), Wednesday (61, 14.3%), Tuesday (60, 14.1%).

**Busiest periods**: 15:00-17:59 (77, 18.0%), 21:00-23:59 (73, 17.1%), 18:00-20:59 (70, 16.4%).

**Offences completed**: (415, 97.2%).

**Top premises**: RESIDENCE - APT. HOUSE (127, 29.7%), RESIDENCE-HOUSE (104, 24.4%), STREET (86, 20.1%).

**Top precincts**: 103 (57, 13.3%), 115 (46, 10.8%), 114 (36, 8.4%).

**Victim** ages: 25-44, 45-64, 18-24; 48.2% male; race 30.9% Black, 12.2% White, 36.1% Hispanic.

**Suspect** ages: 25-44, 45-64, 18-24; 74.4% male; race 38.9% Black, 8.3% White, 31.1% Hispanic.

## Overview of ROBBERY in STATEN ISLAND

There were 8 offences reported.

**Peak days**: Monday (3, 37.5%), Sunday (3, 37.5%), Tuesday (1, 12.5%).

**Busiest periods**: 18:00-20:59 (3, 37.5%), 15:00-17:59 (2, 25.0%), 21:00-23:59 (2, 25.0%).

**Offences completed**: (5, 62.5%).

**Top premises**: STREET (2, 25.0%), CHAIN STORE (2, 25.0%), OTHER (1, 12.5%).

**Top precincts**: 122 (3, 37.5%), 120 (3, 37.5%), 121 (2, 25.0%).

**Victim** ages: 25-44, 45-64, 18-24; 75.0% male; race 37.5% Black, 25.0% White, 12.5% Hispanic.

**Suspect** ages: 25-44, 45-64, <18; 75.0% male; race 37.5% Black, 37.5% White, 12.5% Hispanic.

## Overview of BURGLARY in STATEN ISLAND

There were 20 offences reported.

**Peak days**: Tuesday (5, 25.0%), Friday (4, 20.0%), Wednesday (3, 15.0%).

**Busiest periods**: 18:00-20:59 (4, 20.0%), 21:00-23:59 (4, 20.0%), 00:00-02:59 (3, 15.0%).

**Offences completed**: (20, 100.0%).

**Top premises**: RESIDENCE-HOUSE (10, 50.0%), OTHER (2, 10.0%), PARKING LOT/GARAGE (PRIVATE) (1, 5.0%).

**Top precincts**: 121 (8, 40.0%), 120 (7, 35.0%), 122 (3, 15.0%).

**Victim** ages: 25-44, 45-64, Unknown; 40.0% male; race 10.0% Black, 20.0% White, 10.0% Hispanic.

**Suspect** ages: 25-44, <18, Unknown; 91.7% male; race 25.0% Black, 33.3% White, 11.1% Hispanic.

## Overview of FELONY ASSAULT in STATEN ISLAND

There were 82 offences reported.

**Peak days**: Thursday (18, 22.0%), Sunday (16, 19.5%), Wednesday (13, 15.9%).

**Busiest periods**: 15:00-17:59 (17, 20.7%), 12:00-14:59 (16, 19.5%), 00:00-02:59 (14, 17.1%).

**Offences completed**: (76, 92.7%).

**Top premises**: RESIDENCE-HOUSE (30, 36.6%), RESIDENCE - APT. HOUSE (16, 19.5%), STREET (11, 13.4%).

**Top precincts**: 120 (42, 51.2%), 121 (23, 28.0%), 123 (10, 12.2%).

**Victim** ages: 25-44, <18, 45-64; 41.5% male; race 32.9% Black, 28.0% White, 22.0% Hispanic.

**Suspect** ages: 25-44, 45-64, 18-24; 69.7% male; race 50.0% Black, 23.0% White, 18.9% Hispanic.